## Match to OpenAlex

### What we did

1. **Loaded data**: Read `huang_awards_pilot.csv` (all 31 conferences, including ICWSM + JCDL merged in notebook 03)

2. **Pre-populated already-matched rows**: ICWSM and JCDL papers were matched to OpenAlex in notebook `01b`. We load `icwsm_jcdl_awards_raw.csv` and seed those `openalex_id` values in before the loop so they are skipped entirely — no redundant API calls.

3. **Matched remaining papers to OpenAlex**: Used three matching routes for everything not pre-matched:
    - **Route A (SS→DOI→OA)**: Extracted Semantic Scholar hash from URL → fetched DOI via SS API → looked up in OpenAlex
    - **Route C (SS→title→OA)**: When SS had no DOI, fell back to title+year search in OpenAlex
    - **Route B (GS→title→OA)**: For Google Scholar URLs, directly searched OpenAlex by title+year

4. **API integration**: 
    - Called Semantic Scholar API to extract paper metadata (DOI, title, year)
    - Called OpenAlex API to retrieve paper records and IDs
    - Applied rate limiting (0.1-0.15s delays) to respect API quotas

5. **Quality checks**: 
    - Calculated title similarity scores using fuzzy matching
    - Flagged low-confidence matches (< 85% similarity) for manual review

6. **Output**: 
    - Saved matched results to CSV with OpenAlex IDs, match routes, and confidence scores
    - Exported unmatched papers for manual lookup
    - Printed summary statistics by matching route

In [ ]:
import pandas as pd
import requests
import time
import re
from urllib.parse import quote

df = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\cleaned\\huang_awards_pilot.csv')
MAILTO = 'shaheryar.4822@student.uu.se'

# ── Pre-populate already-matched ICWSM / JCDL rows ─────────────────────────────
# These were matched in notebook 01b; load their openalex_id so we can skip them.
icwsm_jcdl = pd.read_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\raw\\icwsm_jcdl_awards_raw.csv')

# Build a lookup keyed on (normalised title, year)
def norm(t):
    return re.sub(r'\s+', ' ', str(t).lower().strip())

pre_matched = {
    (norm(r['paper_title']), int(r['year'])): r
    for _, r in icwsm_jcdl.iterrows()
    if pd.notna(r.get('openalex_id'))
}
print(f"Pre-matched from 01b: {len(pre_matched)} rows")

# ── Helpers ────────────────────────────────────────────────────────────────────

def extract_ss_hash(url):
    if not isinstance(url, str) or 'semanticscholar' not in url:
        return None
    match = re.search(r'[a-f0-9]{40}', url)
    return match.group(0) if match else None

def ss_to_doi(ss_hash, retries=2):
    url = f"https://api.semanticscholar.org/graph/v1/paper/{ss_hash}?fields=externalIds,title,year"
    for _ in range(retries):
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                data = r.json()
                doi = data.get('externalIds', {}).get('DOI')
                return doi, data.get('title'), data.get('year')
            elif r.status_code == 429:
                time.sleep(5)
        except:
            time.sleep(2)
    return None, None, None

def openalex_by_doi(doi):
    url = f"https://api.openalex.org/works/https://doi.org/{doi}"
    try:
        r = requests.get(url, params={'mailto': MAILTO}, timeout=10)
        if r.status_code == 200:
            return r.json()
    except:
        pass
    return None

def openalex_by_title(title, year):
    clean = re.sub(r"['\"-\/\\]", ' ', title).strip()
    clean = re.sub(r'\s+', ' ', clean)
    params = {
        'search': clean,
        'filter': f'publication_year:{year}',
        'per-page': 3,
        'mailto': MAILTO
    }
    try:
        r = requests.get('https://api.openalex.org/works', params=params, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results:
                return results[0]
    except:
        pass
    return None

# ── Main matching loop ──────────────────────────────────────────────────────────

records = []

for i, row in df.iterrows():
    url = str(row.get('paper_url', ''))
    title = str(row.get('paper_title', ''))
    year = int(row.get('year', 0))
    result = {
        'year': year, 'conference': row['conference'],
        'paper_title': title, 'paper_url': url,
        'authors': row.get('authors', ''),
        'openalex_id': None, 'doi': None,
        'match_route': None, 'oa_title': None
    }

    # ── Skip if already matched in 01b ──
    key = (norm(title), year)
    if key in pre_matched:
        pm = pre_matched[key]
        result['openalex_id'] = pm.get('openalex_id')
        result['doi']         = pm.get('doi')
        result['oa_title']    = pm.get('openalex_title')
        result['authorships'] = pm.get('authorships', '')
        result['match_route'] = 'pre-matched'
        records.append(result)
        continue

    oa_work = None

    # ── Route A: Semantic Scholar hash ──
    ss_hash = extract_ss_hash(url)
    if ss_hash:
        doi, ss_title, ss_year = ss_to_doi(ss_hash)
        time.sleep(0.15)

        if doi:
            oa_work = openalex_by_doi(doi)
            result['doi'] = doi
            result['match_route'] = 'SS→DOI→OA'

        if not oa_work:
            search_title = ss_title or title
            oa_work = openalex_by_title(search_title, year)
            result['match_route'] = 'SS→title→OA'

    # ── Route B: Google Scholar → title search ──
    else:
        oa_work = openalex_by_title(title, year)
        result['match_route'] = 'GS→title→OA'

    # ── Store result ──
    if oa_work:
        result['openalex_id'] = oa_work.get('id')
        result['oa_title'] = oa_work.get('title')
        result['authorships'] = str(oa_work.get('authorships', []))
    
    records.append(result)

    if i % 50 == 0:
        matched = sum(1 for r in records if r['openalex_id'])
        print(f"[{i}/{len(df)}] Matched: {matched} | Route A: {sum(1 for r in records if r.get('match_route','').startswith('SS→DOI'))}")
    
    time.sleep(0.1)

# ── Save ────────────────────────────────────────────────────────────────────────
out = pd.DataFrame(records)
out.to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\huang_matched_openalex.csv', index=False)

matched = out['openalex_id'].notna().sum()
print(f"\n{'='*50}")
print(f"Total: {len(out)} | Matched: {matched} ({matched/len(out)*100:.1f}%)")
print(out.groupby('match_route')['openalex_id'].apply(lambda x: x.notna().sum()))


#### Check

In [ ]:
from rapidfuzz import fuzz
# Flag low-confidence matches — pre-matched rows get similarity 100 automatically
out['title_similarity'] = out.apply(
    lambda r: 100 if r['match_route'] in ('SS→DOI→OA', 'pre-matched')
    else fuzz.ratio(str(r['paper_title']).lower(), str(r['oa_title']).lower()),
    axis=1
)
out['low_confidence'] = out['title_similarity'] < 85
print(out['low_confidence'].sum(), "flagged for manual review")


#### Manual Checkups

In [ ]:
unmatched = out[out['openalex_id'].isna()]
unmatched[['year','conference','paper_title']].to_csv('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\unmatched_manual.csv', index=False)
